# LISS-4 ClearNet: Git-Integrated Cloud Removal Pipeline
### Beyond the Clouds Â· BAH 2026

This notebook clones your GitHub repository directly into Colab, downloads the dataset, and runs training/evaluation.

## 1. Environment Setup & Mount Google Drive

In [ ]:
# Click the 'Mount Drive' folder icon in the left file panel if this code block throws a credential error
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install -q rasterio scikit-image pystac-client requests tqdm wandb

## 2. Clone Your GitHub Repository

In [ ]:
# Clone your repository
!git clone https://github.com/Vickyrrrrrr/bah2026-proposal.git /content/bah2026-proposal

import sys
sys.path.append('/content/bah2026-proposal/src')

## 3. Download Dataset Split via Rsync

In [ ]:
# Create local fast storage folders in Colab
!mkdir -p /content/data/ROIs1158_spring

# Download the three spring archives (SAR, Cloudy, Ground-truth) directly to Colab local disk (~70 GB total)
!export RSYNC_PASSWORD=m1554803 && rsync -chavzP \
  rsync://m1554803@dataserv.ub.tum.de/m1554803/ROIs1158_spring_s1.tar.gz \
  rsync://m1554803@dataserv.ub.tum.de/m1554803/ROIs1158_spring_s2_cloudy.tar.gz \
  rsync://m1554803@dataserv.ub.tum.de/m1554803/ROIs1158_spring_s2.tar.gz \
  /content/

# Extract the archives locally
print("Extracting Sentinel-1 SAR...")
!tar -xzf /content/ROIs1158_spring_s1.tar.gz -C /content/data/ROIs1158_spring/
print("Extracting Cloudy Sentinel-2...")
!tar -xzf /content/ROIs1158_spring_s2_cloudy.tar.gz -C /content/data/ROIs1158_spring/
print("Extracting Ground-truth Sentinel-2...")
!tar -xzf /content/ROIs1158_spring_s2.tar.gz -C /content/data/ROIs1158_spring/

# Remove downloaded archives to free up local disk space
!rm /content/*.tar.gz
print("✅ Dataset ready locally!")

## 4. Execute Training

In [ ]:
# Run training using the local fast dataset, but saving model checkpoints directly to Google Drive
!python /content/bah2026-proposal/src/train.py \
  --data_dir /content/data/ \
  --epochs 50 \
  --batch_size 8 \
  --output_dir /content/drive/MyDrive/LISS4_CloudRemoval/checkpoints/